In [ ]:
# 导入必要的库
import os
import yaml
from agents.retrieval_agent import RetrievalAgent
from agents.response_generator_agent import ResponseGeneratorAgent
from agents.response_validator_agent import ResponseValidatorAgent
from agents.response_selector_agent import ResponseSelectorAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo
import os
from pathlib import Path
import yaml
from autogen_ext.memory.chromadb import ChromaDBVectorMemory, PersistentChromaDBVectorMemoryConfig,CustomEmbeddingFunctionConfig
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo
# 加载配置
with open('settings.yaml', 'r') as f:
    config = yaml.safe_load(f)

llm_config = config['llm']
model_client = OpenAIChatCompletionClient(
        model=llm_config['model'],
        model_info=ModelInfo(vision=False, function_calling=True, json_output=True, family="unknown", structured_output=True),
        api_key=os.environ.get('OPENAI_API_KEY', llm_config['api_key']),
        base_url=llm_config['base_url'],
)
# Initialize vector memory
vector_store_config = PersistentChromaDBVectorMemoryConfig(
        collection_name=config['vector_store']['collection_name'],
        persistence_path=os.path.expandvars(config['vector_store']['persistence_path'].replace('${HOME}', str(Path.home()))),
        k=config['vector_store']['k'],
        score_threshold=config['vector_store']['score_threshold'],
)
    
# Check if custom embedding function is enabled in config
if config['vector_store'].get('embedding', {}).get('use_custom', False):
        def create_openai_embedding_function(api_key, model, api_base):
            from chromadb.utils import embedding_functions
            return embedding_functions.OpenAIEmbeddingFunction(
                api_key=api_key,
                model_name=model,
                api_base=api_base
            )
        
        embedding_config = config['vector_store']['embedding']
        params = {
            "api_key": embedding_config['api_key'],
            "model": embedding_config['model'],
            "api_base": embedding_config['api_base']
        }
        
        vector_store_config.embedding_function_config = CustomEmbeddingFunctionConfig(
            function=create_openai_embedding_function,
            params=params
        )
    
memory = ChromaDBVectorMemory(config=vector_store_config)

In [ ]:
import asyncio
import json
import sys
from typing import Any, Dict
from asyncio import Queue
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from datetime import datetime

# 获取当前本地时间
now = datetime.now()
print(now.strftime("%Y-%m-%d %H:%M:%S"))  # 输出: 2025-11-07 20:30:45

input_file = "../MultiHopRAG.json"
output_file = "processed_results_test.jsonl"

async def create_team_pool(size: int) -> Queue:
    pool = Queue()
    for i in range(size):
        text_termination = TextMentionTermination("TERMINATE:")
        retrieval_agent = RetrievalAgent(
            name='retrieval_agent',
            model_client=model_client,
            memory=memory,
            retrieval_num=config['vector_store']['k'],
            index_path=config['vector_store']['bm25_index_path']
        )
        generator_agent = ResponseGeneratorAgent(
            name='generator_agent',
            model_client=model_client,
            memory=memory,
            num_versions=config['rag']['num_versions']# 生成的答案的版本数量
        )
        validator_agent = ResponseValidatorAgent(
            name='validator_agent',
            model_client=model_client,
            count=3 # 验证器对每个答案的验证次数
        )
        selector_agent = ResponseSelectorAgent(
            name='selector_agent',
            model_client=model_client,
            memory=memory,
        )
        team = RoundRobinGroupChat(
            [retrieval_agent, generator_agent, selector_agent, validator_agent],
            termination_condition=text_termination
        )
        pool.put_nowait(team)  # 放入池中
    return pool

# ✅ 处理单条数据，使用池中的 team
async def process_item_with_pool(
    data: Dict[str, Any],
    line_num: int,
    team_pool: Queue
) -> Dict[str, Any]:
    team = None
    try:
        # 从池中获取一个 team（会被自动等待，直到有空闲）
        team = await team_pool.get()
        query = data.get("query")
        if not query:
            return {"line_num": line_num, "error": "缺少 'query'", "status": "skipped"}

        # 使用这个 team 执行任务
        result = await team.run(task=query)
        messages = result.messages
        retrival_contexts = []
        generated_answers_list=[]
        selected_answers=[]
        validator_answer=""
        for message in messages:
            if message.source == 'retrieval_agent':
                retrival_task = message.content
                retrival_task_data = json.loads(retrival_task)
                retrieval_context = retrival_task_data["retrieval_context"]
                retrival_contexts.append(retrieval_context)
            elif message.source == 'generator_agent':
                generator_task = message.content
                generator_task_data = json.loads(generator_task)
                generated_answers = generator_task_data["generated_answers"]
                generated_answers_list.append(generated_answers)
            elif message.source == 'selector_agent':
                selector_task = message.content
                selector_task_data = json.loads(selector_task)
                selected_answer = selector_task_data["answer"]
                selected_answers.append(selected_answer)
            elif message.source == 'validator_agent':
                validator_answer = message.content[10:]
        return {
            "query": query,
            "answer": data.get("answer"),
            "question_type": data.get("question_type"),
            "evidence_list": data.get("evidence_list"),
            "retrival_contexts": retrival_contexts,
            "generated_answers_list": generated_answers_list,
            "selected_answers": selected_answers,
            "validator_answer": validator_answer,
            "line_num": line_num,
            "status": "success"
        }

    except Exception as e:
        return {
            "line_num": line_num,
            "error": str(e),
            "query": data.get("query"),
            "status": "error"
        }
    finally:
        # ✅ 无论成功失败，都要把 team 放回池中，供后续使用
        if team:
            await team_pool.put(team)

async def main():
    # 1. 读取输入数据
    try:
        with open(input_file, "r", encoding="utf-8") as fin:
            data_list = json.load(fin)
    except Exception as e:
        print(f"❌ 读取输入文件失败: {e}")
        sys.exit(1)

    # 2. 读取已有输出文件，获取已处理的 line_num
    processed_line_nums = set()
    if os.path.exists(output_file):
        try:
            with open(output_file, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        try:
                            item = json.loads(line)
                            line_num = item.get("line_num")
                            if line_num is not None:
                                processed_line_nums.add(line_num)
                        except json.JSONDecodeError:
                            continue  # 跳过损坏的行
            print(f"🟡 已从 {output_file} 恢复 {len(processed_line_nums)} 条记录。")
        except Exception as e:
            print(f"⚠️ 读取输出文件失败，将重新开始: {e}")

    # 3. 创建 team 池
    team_pool = await create_team_pool(3)  # 注意：async queue 需要 asyncio.Queue()

    # 4. 创建任务：只处理未完成的行
    tasks = [
        process_item_with_pool(data, line_num, team_pool)
        for line_num, data in enumerate(data_list, start=1)
        if data and line_num not in processed_line_nums
    ]

    print(f"🚀 共 {len(data_list)} 条数据，已处理 {len(processed_line_nums)} 条，剩余 {len(tasks)} 条待处理...")

    # 5. 并发执行，追加写入结果
    mode = "a" if processed_line_nums else "w"  # 如果已有结果，用追加模式
    with open(output_file, mode, encoding="utf-8") as fout:
        for coro in asyncio.as_completed(tasks):
            result = await coro
            try:
                fout.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")
                fout.flush()

                line_num = result["line_num"]
                if result["status"] == "error":
                    print(f"❌ 第 {line_num} 行出错: {result['error']}")
            except Exception as e:
                print(f"❌ 写入结果失败: {e}")

    print(f"🎉 所有任务完成，结果保存至 {output_file}")
# 🚀 运行
if __name__ == "__main__":
    await main()